In [1]:
# train.py

# Standard library
import argparse
import glob
import json
import os
import shutil
import sys
from tqdm import tqdm
from dataclasses import dataclass
from datetime import datetime
from functools import partial
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional

# Data processing
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset as TorchDataset
from torch.utils.data import DataLoader

# Hugging Face datasets and evaluation
import evaluate
from datasets import (
    Dataset as HFDataset,
    concatenate_datasets,
    load_dataset,
)

# Hugging Face Transformers
from transformers import (
    AutoModel,
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_utils import get_last_checkpoint

# Training and parameter-efficient fine-tuning
from accelerate import Accelerator
from peft import LoraConfig, get_peft_model

# Experiment tracking
import wandb

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
)

In [2]:
#constants
PROBE_WEIGHTS_DIR = './model/probes/'
EVAL_OUT_DIR = "./tests"
os.makedirs(EVAL_OUT_DIR, exist_ok=True)
RANDOM_SEED = 42
GIANT_RESULTS_DICT = './giant_res.csv'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [3]:
odf = pd.read_csv("full_babe_with_cats.csv")
odf['text'] = odf['text'].astype(str).fillna("")
odf["followed_news_outlets_clean"] = (
    odf["followed_news_outlets"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.replace("[", "", regex=False)
    .str.replace("]", "", regex=False)
    .str.replace("'", "", regex=False)
    .str.replace('"', "", regex=False)
)
odf["num_foll_news_outlet"] = odf["followed_news_outlets_clean"].apply(
    lambda x: len([item for item in x.split(",") if item.strip() != ""])
)

odf["num_age"] = pd.cut(odf["age"],bins=[-np.inf, 18, 30, 45, 55, 65, np.inf],labels=False)

odf["num_poli_idealogy"] = pd.cut(odf["political_ideology"],
                                 bins=[-np.inf, -6, -1, 1, 5, np.inf],
                                 labels=False)

odf["num_foll_news_outlet"] = pd.cut(odf["num_foll_news_outlet"],
                                          bins=[-np.inf, 1, 2, 3, 4, 6, np.inf],
                                          labels=False)


odf["num_news_check_frequency"] = odf["news_check_frequency"].map({
    "Never": 0,
    "Very rarely": 1,
    "Several times per month": 2,
    "Several times per week": 3,
    "Every day": 4,
    "Several times per day": 5,
})

def bin_education(edu):
    edu = str(edu).strip()
    if edu in {"Graduate work"}:
        return 5
    if edu in {"BA", "Bachelor", "Bachelors", "Bachelor's degree", "Bachelor’s degree"}:
        return 4
    if edu in {"Associate degree"}:
        return 3
    if edu in {"Some college", "Vocational or technical school"}:
        return 2
    if edu == "High school graduate":
        return 1
    return 0

odf["num_education"] = odf["education"].apply(bin_education)
odf['num_gender'] = odf['gender'].map({
    'Male': 1, 'male': 1,
    'Female': 0, 'female': 0,
}).fillna(2).astype(int)
odf["label_binary"] = odf["label_bias"].str.startswith("Biased").astype(int)

def binary_entropy(x):
    p = x.mean()
    if p == 0 or p == 1:
        return 0.0
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

# odf['text'].nunique #49733
# odf['sentence_id'].nunique #49733 -same

entropy_df = (
    odf.groupby("sentence_id")["label_binary"]
      .agg(
          n="count",
          p_1="mean",
          entropy=binary_entropy
      )
      .reset_index()
)
odf = odf.merge(entropy_df[["sentence_id", "entropy"]], on="sentence_id", how="left")

/scratch/slurm-3164687/ipykernel_3453994/2991592148.py:1: DtypeWarning: Columns (0: survey_completed) have mixed types. Specify dtype option on import or set low_memory=False.
  odf = pd.read_csv("full_babe_with_cats.csv")


In [4]:
#embeds

cat_cardinalities = {
    "words": 11267,
    "factual": 3,
    "group_id": 85,
    "type": 3,
    "topic": 14,
    "outlet": 8,
    "age": 54,
    "gender": 3,
    "education": 8,
    "native_english_speaker": 3,
    "political_ideology": 21,
    "followed_news_outlets": 551,
    "news_check_frequency": 6,
    "num_news_check_frequency": 6,
    "num_foll_news_outlet": 6,
    "num_poli_idealogy": 6,
    "num_age": 6,
    "num_gender": 3,
    "num_education": 6,
    
}



out_cats = ["num_age", "num_gender", "num_education", "num_poli_idealogy", "num_news_check_frequency", "num_foll_news_outlet"]
in_cats = [k for k, card in cat_cardinalities.items() if k not in ["num_age", "num_gender", "num_education", "num_poli_idealogy", "num_news_check_frequency", "num_foll_news_outlet"]]
cat_embs_info = {}
for k, card in cat_cardinalities.items():
    if k not in odf.columns:
        continue
    emb_dim = max(2, int(card ** 0.5) + 1) #sqrt(cardinality)) + 1
    emb_dim = min(50, emb_dim) #but not more than 50 - cough cough words
    cat_embs_info[k] = {
        "cardinality": card, 
        "emb_dim": emb_dim,
        "out_cat": k in out_cats,
        "in_cat": k in in_cats}




class MTLDataset(TorchDataset):
    def __init__(self, 
                 df, 
                 cat_info, 
                 tokenizer,
                 zero_cats = False,
                 max_length=128):
        
        df = df.reset_index(drop=True).copy()
        self.texts = df["text"].tolist()
     
        for col in list(cat_info.keys()):
            df[col] = df[col].astype('category')
            try:
                df[col] = df[col].cat.codes
            except:
                print(col)

        cat_ids = df[list(cat_info.keys())].values   #[N, num_cat_vars]
        cat_ids = cat_ids.astype(int)

        if zero_cats:
            cat_ids = np.zeros_like(cat_ids)

    
        self.cat_ids = cat_ids
        self.labels = df["label_binary"].to_numpy(dtype=np.float32)
        self.entrops = df["entropy"].to_numpy(dtype=np.float32)
        self.out_cat_names = [k for k, v in cat_info.items() if v.get("out_cat", False)]
        self.out_cat_ids = {name: df[name].values for name in self.out_cat_names}
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        cats = self.cat_ids[idx]

        enc = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "x_cats": torch.tensor(cats, dtype=torch.long),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float),
            "entrops": torch.tensor(self.entrops[idx], dtype=torch.float),
            
            "target_cats": torch.tensor(
                [self.out_cat_ids[name][idx] for name in self.out_cat_names],
                dtype=torch.long
            ),
        }
        return item
    
    


In [5]:
class ambiProbe(nn.Module):

    
    def __init__(self, 
                cats_info, 
                llm_pretrained_name, 
                parameters
                ):
        super().__init__()

        self.condEntropyWeight = parameters.get('condEntropyWeight')

        self.llm = AutoModel.from_pretrained(
            llm_pretrained_name,
            output_hidden_states=True
        )
        for p in self.llm.parameters():
            p.requires_grad = False
        
        self.llm.eval()

                

        d_model = self.llm.config.hidden_size

        late_fusion_width = d_model + 1 #plus 1 for the label
        self.linear_post_fusion = nn.Linear(late_fusion_width, d_model)

        
        self.shared_trunk = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, d_model // 4),
        )
        
        self.out_cat_names = [k for k, v in cats_info.items() if v.get("out_cat", False)]
        self.probe_heads = nn.ModuleDict({
            name: nn.Linear(d_model // 4, cats_info[name]["cardinality"])
            for name in self.out_cat_names
        })

    def forward(
        self,
        input_ids, #embedding of input text
        attention_mask,
        x_cats,
        labels, # [B] binary silver labels
        target_cats,  # [B, out_cats] numerical features
        entrops = None #[B] entropies per sentence
    ):

        with torch.no_grad():
            outputs = self.llm(input_ids=input_ids, attention_mask=attention_mask)
        #uncollate here instead of building a collator for the dataset?
        target_cats_dict = {}
        for idx, name in enumerate(self.out_cat_names):
            target_cats_dict[name] = target_cats[:, idx]

        last_hidden = outputs.last_hidden_state # [B, L, d_model]
        mask = attention_mask.unsqueeze(-1)            # [B, L, 1]
        hidden_masked = last_hidden * mask             # [B, L, d_model]
        sum_hidden = hidden_masked.sum(dim=1)          # [B, d_model]
        lengths = mask.sum(dim=1).clamp(min=1)         # [B, 1]
        h_text = sum_hidden / lengths                  # [B, d_model]
        

        h_fused = torch.cat([h_text, labels.unsqueeze(-1).float()], dim=-1) # [B, d_model + 1]
        h_proj = self.linear_post_fusion(h_fused)    # [B, d_model]
        h_shared = self.shared_trunk(h_proj)         # [B, shared_hdim]

        

        logits = {name: head(h_shared) for name, head in self.probe_heads.items()}

        for name in self.probe_heads:
            target = target_cats_dict[name]
            logit = logits[name]
            n_classes = logit.shape[-1]


            bad = (target < 0) | (target >= n_classes)
            if bad.any():
                bad_vals = target[bad].detach().cpu().tolist()[:20]
                raise ValueError(
                    f"Bad targets for head {name}: {bad_vals}; "
                    f"valid range is [0, {n_classes - 1}]"
                )


                
        losses = {name: F.cross_entropy(logits[name], target_cats_dict[name], reduction="none") # returns [B] — one loss per row
                  for name in self.probe_heads}
        #stack the piecewise losses
        losses = torch.stack(list(losses.values())) #makes a cats x B
        #sum on the vertical
        losses = losses.sum(dim=0) #now a [1 x B]
        if entrops is not None:
            weighted_entrops = entrops.to(losses.device).float()  * self.condEntropyWeight
            cond_losses = losses * weighted_entrops
            total_loss = cond_losses.sum() / weighted_entrops.sum().clamp(min=1e-8)
        else:
            cond_losses = losses
            total_loss = cond_losses.sum()
            


        return {
            "loss": total_loss,
            "piecewise_losses": losses,
            "out_logits" : logits
        }
        

## GO-NO GO TEST
Here we'll test the first tinyLLama, run for 3 epochs, on both the random lables and random targets. we'll try to build this in a principled manner, as we'll repeat these tests for others. however, this code block is strictly making sure we're in the right direction first. 

In [6]:
#bookeeping
def make_shuffle_columns_transform(dataset,same_permutation=True):
    """
    same_permutation=True preserves whole demographic profiles.
    same_permutation=False shuffles each target head independently.
    """
    columns = out_cats

    def transform(df):
        out = df.copy()
        rng = np.random.default_rng(RANDOM_SEED)

        if same_permutation:
            perm = rng.permutation(len(out))
            for col in columns:
                out[col] = out[col].to_numpy()[perm]
        else:
            for col in columns:
                perm = rng.permutation(len(out))
                out[col] = out[col].to_numpy()[perm]

        return out

    return transform


def quartile_transform(dataset, frac=0.25, entropy_col = "entrops"):
    out = df.copy()
    n = max(1, int(np.ceil(frac * len(out))))
    keep_idx = out[entropy_col].nlargest(n).index
    return out.loc[keep_idx].sort_index().copy()

In [7]:



def _save_predictions_csv(
    path,
    y_true,
    y_pred,
    out_cat_names,
    entropies,
):
    prediction_df = pd.DataFrame({
        "row_position": np.arange(len(entropies)),
        "entropy": entropies,
    })

    for name in out_cat_names:
        prediction_df[f"{name}__gold"] = y_true[name]
        prediction_df[f"{name}__pred"] = y_pred[name]

    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    prediction_df.to_csv(path, index=False)


def _load_predictions_csv(
    path,
    out_cat_names,
    expected_entropies,
):
    prediction_df = pd.read_csv(path)

    required_columns = ["row_position", "entropy"]
    expected_positions = np.arange(len(expected_entropies))
    saved_positions = prediction_df["row_position"].to_numpy()
    saved_entropies = prediction_df["entropy"].to_numpy(dtype=float)

    prediction_columns = [
        f"{name}__gold"
        for name in out_cat_names
    ] + [
        f"{name}__pred"
        for name in out_cat_names
    ]
    y_true = {
        name: prediction_df[f"{name}__gold"].to_numpy(dtype=np.int64)
        for name in out_cat_names
    }

    y_pred = {
        name: prediction_df[f"{name}__pred"].to_numpy(dtype=np.int64)
        for name in out_cat_names
    }

    return y_true, y_pred

In [8]:
# Top 25% means rows at or above the 75th percentile.
HIGH_ENTROPY_PERCENTILE = 0.75


def _nan_if_no_support(f1s, supports):
    return [
        np.nan if support == 0 else float(f1)
        for f1, support in zip(f1s, supports)
    ]




def _calculate_head_metrics(
    model,
    out_cat_names,
    y_true,
    y_pred,
    row_mask=None,
):
    by_head = {}

    for name in out_cat_names:
        gold = y_true[name]
        pred = y_pred[name]

        if row_mask is not None:
            gold = gold[row_mask]
            pred = pred[row_mask]

        if len(gold) == 0:
            raise ValueError("No rows were selected for metric calculation.")

        n_classes = model.probe_heads[name].out_features
        all_labels = np.arange(n_classes)

        counts = np.bincount(gold, minlength=n_classes)
        modal_class = int(np.argmax(counts))
        baseline_pred = np.full_like(gold, modal_class)

        present_labels = np.flatnonzero(counts > 0)

        probe_acc = accuracy_score(gold, pred)
        base_acc = accuracy_score(gold, baseline_pred)

        probe_macro_f1 = f1_score(
            gold,
            pred,
            labels=present_labels,
            average="macro",
            zero_division=0,
        )

        base_macro_f1 = f1_score(
            gold,
            baseline_pred,
            labels=present_labels,
            average="macro",
            zero_division=0,
        )

        _, _, probe_f1s, supports = precision_recall_fscore_support(
            gold,
            pred,
            labels=all_labels,
            zero_division=0,
        )

        _, _, base_f1s, _ = precision_recall_fscore_support(
            gold,
            baseline_pred,
            labels=all_labels,
            zero_division=0,
        )

        probe_f1s = _nan_if_no_support(probe_f1s, supports)
        base_f1s = _nan_if_no_support(base_f1s, supports)

        by_head[name] = {
            "n": int(len(gold)),

            "accuracy": float(probe_acc),
            "baseline_accuracy": float(base_acc),
            "accuracy_lift": float(probe_acc - base_acc),

            "macro_f1": float(probe_macro_f1),
            "baseline_macro_f1": float(base_macro_f1),
            "macro_f1_lift": float(probe_macro_f1 - base_macro_f1),

            "modal_class": modal_class,

            "support_by_class": {
                int(cls): int(support)
                for cls, support in zip(all_labels, supports)
            },
            "f1_by_class": {
                int(cls): f1
                for cls, f1 in zip(all_labels, probe_f1s)
            },
            "baseline_f1_by_class": {
                int(cls): f1
                for cls, f1 in zip(all_labels, base_f1s)
            },
        }

    return by_head


def _calculate_global_metrics(by_head, n_rows):
    return {
        "n_rows": int(n_rows),
        "n_heads": int(len(by_head)),

        # Average over heads, not over individual predictions.
        "accuracy": float(
            np.mean([m["accuracy"] for m in by_head.values()])
        ),
        "baseline_accuracy": float(
            np.mean([m["baseline_accuracy"] for m in by_head.values()])
        ),
        "accuracy_lift": float(
            np.mean([m["accuracy_lift"] for m in by_head.values()])
        ),

        "macro_f1": float(
            np.mean([m["macro_f1"] for m in by_head.values()])
        ),
        "baseline_macro_f1": float(
            np.mean([m["baseline_macro_f1"] for m in by_head.values()])
        ),
        "macro_f1_lift": float(
            np.mean([m["macro_f1_lift"] for m in by_head.values()])
        ),

        "global_average": "macro_over_heads",
    }

@torch.inference_mode()
def evaluate_raw(
    model,
    dataset,
    test_name,
    batch_size=64,
    device=None,
    predictions_csv_path=None,
):

    model.eval()

    out_cat_names = dataset.out_cat_names

    entropies = np.asarray(dataset.entrops, dtype=float)

    # ---------------------------------------------------------
    # Either load saved predictions or run model inference
    # ---------------------------------------------------------
    run_eval_dir = os.path.join(EVAL_OUT_DIR, test_name)
    os.makedirs(run_eval_dir, exist_ok=True)
    preds_path = os.path.join(run_eval_dir, "predictions.csv")
    
    if predictions_csv_path is not None:
        #preds should go in a subfolder of EVAL_OUT_DIR, labeled with the run_name text passed in
        
        y_true, y_pred = _load_predictions_csv(
            path=predictions_csv_path,
            out_cat_names=out_cat_names,
            expected_entropies=entropies,
        )

        prediction_source = "csv"
        used_predictions_path = str(predictions_csv_path)

    else:
        if device is None:
            device = next(model.parameters()).device

        y_true = {
            name: []
            for name in out_cat_names
        }

        y_pred = {
            name: []
            for name in out_cat_names
        }

        loader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=0,
        )

        for batch in tqdm(loader, desc="Evaluating probe"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            x_cats = batch["x_cats"].to(device)
            target_cats = batch["target_cats"].to(device)

            out = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                x_cats=x_cats,
                labels=labels,
                target_cats=target_cats,
                entrops=None,
            )

            for cat_idx, name in enumerate(out_cat_names):
                preds = out["out_logits"][name].argmax(dim=-1)
                golds = target_cats[:, cat_idx]

                y_pred[name].append(preds.cpu().numpy())
                y_true[name].append(golds.cpu().numpy())

        y_true = {
            name: np.concatenate(values)
            for name, values in y_true.items()
        }

        y_pred = {
            name: np.concatenate(values)
            for name, values in y_pred.items()
        }

        _save_predictions_csv(
            path=preds_path,
            y_true=y_true,
            y_pred=y_pred,
            out_cat_names=out_cat_names,
            entropies=entropies,
        )

        prediction_source = "inference"
        used_predictions_path =preds_path

    # ---------------------------------------------------------
    # All dataset rows
    # ---------------------------------------------------------
    by_head = _calculate_head_metrics(
        model=model,
        out_cat_names=out_cat_names,
        y_true=y_true,
        y_pred=y_pred,
    )

    global_metrics = _calculate_global_metrics(
        by_head=by_head,
        n_rows=len(dataset),
    )

    # ---------------------------------------------------------
    # Highest-entropy rows
    # ---------------------------------------------------------
    entropy_threshold = np.nanquantile(
        entropies,
        HIGH_ENTROPY_PERCENTILE,
    )

    high_entropy_mask = (
        np.isfinite(entropies)
        & (entropies >= entropy_threshold)
    )

    high_entropy_by_head = _calculate_head_metrics(
        model=model,
        out_cat_names=out_cat_names,
        y_true=y_true,
        y_pred=y_pred,
        row_mask=high_entropy_mask,
    )

    high_entropy_global = _calculate_global_metrics(
        by_head=high_entropy_by_head,
        n_rows=high_entropy_mask.sum(),
    )

    return {
        "global": global_metrics,
        "by_head": by_head,

        "high_entropy": {
            "percentile": float(HIGH_ENTROPY_PERCENTILE),
            "threshold": float(entropy_threshold),
            "n_rows": int(high_entropy_mask.sum()),
            "global": high_entropy_global,
            "by_head": high_entropy_by_head,
        },

        "prediction_cache": {
            "source": prediction_source,
            "path": used_predictions_path,
        },
    }


def print_eval_summary(result):
    def print_section(by_head, global_metrics):
        print(
            f"{'Variable':<29}"
            f"{'Probe':>9}"
            f"{'Marginal':>11}"
            f"{'Lift':>9}"
        )
        print("-" * 58)

        for name, metrics in by_head.items():
            print(
                f"{name:<29}"
                f"{metrics['macro_f1']:>9.3f}"
                f"{metrics['baseline_macro_f1']:>11.3f}"
                f"{metrics['macro_f1_lift']:>+9.3f}"
            )

        print("-" * 58)
        print(
            f"{'GLOBAL':<29}"
            f"{global_metrics['macro_f1']:>9.3f}"
            f"{global_metrics['baseline_macro_f1']:>11.3f}"
            f"{global_metrics['macro_f1_lift']:>+9.3f}"
        )

    # Existing full-dataset table.
    print_section(
        result["by_head"],
        result["global"],
    )

    high = result["high_entropy"]
    top_percent = 100 * (1 - high["percentile"])

    print()
    print(
        f"HIGHEST {top_percent:.0f}% ENTROPY "
        f"(n={high['n_rows']}, cutoff={high['threshold']:.4f})"
    )

    print_section(
        high["by_head"],
        high["global"],
    )


def one_row_summary(result, model_name, condition):
    """
    One row containing the full-data and high-entropy summaries.
    """
    g = result["global"]

    row = {
        "model": model_name,
        "condition": condition,

        "accuracy": g["accuracy"],
        "baseline_accuracy": g["baseline_accuracy"],
        "accuracy_lift": g["accuracy_lift"],

        "macro_f1": g["macro_f1"],
        "baseline_macro_f1": g["baseline_macro_f1"],
        "macro_f1_lift": g["macro_f1_lift"],

        "global_average": g["global_average"],
    }

    for head, m in result["by_head"].items():
        row[f"{head}__accuracy"] = m["accuracy"]
        row[f"{head}__baseline_accuracy"] = m["baseline_accuracy"]
        row[f"{head}__accuracy_lift"] = m["accuracy_lift"]

        row[f"{head}__macro_f1"] = m["macro_f1"]
        row[f"{head}__baseline_macro_f1"] = m["baseline_macro_f1"]
        row[f"{head}__macro_f1_lift"] = m["macro_f1_lift"]

        row[f"{head}__modal_class"] = m["modal_class"]

    high = result["high_entropy"]
    hg = high["global"]

    row.update({
        "high_entropy__percentile": high["percentile"],
        "high_entropy__threshold": high["threshold"],
        "high_entropy__n_rows": high["n_rows"],

        "high_entropy__accuracy": hg["accuracy"],
        "high_entropy__baseline_accuracy": hg["baseline_accuracy"],
        "high_entropy__accuracy_lift": hg["accuracy_lift"],

        "high_entropy__macro_f1": hg["macro_f1"],
        "high_entropy__baseline_macro_f1": hg["baseline_macro_f1"],
        "high_entropy__macro_f1_lift": hg["macro_f1_lift"],
    })

    for head, m in high["by_head"].items():
        prefix = f"high_entropy__{head}"

        row[f"{prefix}__accuracy"] = m["accuracy"]
        row[f"{prefix}__baseline_accuracy"] = m["baseline_accuracy"]
        row[f"{prefix}__accuracy_lift"] = m["accuracy_lift"]

        row[f"{prefix}__macro_f1"] = m["macro_f1"]
        row[f"{prefix}__baseline_macro_f1"] = m["baseline_macro_f1"]
        row[f"{prefix}__macro_f1_lift"] = m["macro_f1_lift"]

        row[f"{prefix}__modal_class"] = m["modal_class"]

    return row

def _json_safe(obj):
    """
    Converts numpy scalars / NaNs into JSON-safe values.
    """
    if isinstance(obj, dict):
        return {
            str(k): _json_safe(v)
            for k, v in obj.items()
        }

    if isinstance(obj, list):
        return [
            _json_safe(v)
            for v in obj
        ]

    if isinstance(obj, tuple):
        return [
            _json_safe(v)
            for v in obj
        ]

    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.floating):
        if np.isnan(obj):
            return None

        return float(obj)

    if isinstance(obj, float) and np.isnan(obj):
        return None

    return obj


def write_eval_log(
    result,
    path,
    model_name,
    condition,
    extra=None,
):
    """
    Saves the full breakdown, including per-class F1/supports.
    """
    payload = {
        "model": model_name,
        "condition": condition,
        "extra": extra or {},
        "result": result,
    }

    payload = _json_safe(payload)

    os.makedirs(
        os.path.dirname(path),
        exist_ok=True,
    )

    with open(path, "w") as f:
        json.dump(
            payload,
            f,
            indent=2,
        )

def print_accuracy_section(by_head, global_metrics, title=None):
    if title is not None:
        print(f"\n{title}")

    print(
        f"{'Variable':<25} "
        f"{'Probe':>8}  "
        f"{'Marginal':>10}  "
        f"{'Lift':>8}"
    )
    print("─" * 55)

    for name, m in by_head.items():
        print(
            f"{name:<25} "
            f"{m['accuracy']:>8.3f}  "
            f"{m['baseline_accuracy']:>10.3f}  "
            f"{m['accuracy_lift']:>+8.3f}"
        )

    print("─" * 55)

    print(
        f"{'GLOBAL':<25} "
        f"{global_metrics['accuracy']:>8.3f}  "
        f"{global_metrics['baseline_accuracy']:>10.3f}  "
        f"{global_metrics['accuracy_lift']:>+8.3f}"
    )

In [49]:
#Test 1 - standard tinyllama 3 epoch (conditional entropy, no other changes), evaluated on f1/acc and lift
run_name = "TinyLlama_probe_3epoch_standard"
model_name = "TinyLlama_probe_3epoch_standard"
condition  = "none"
log_path = f"tests/{run_name}/full_log.json"

output_dir = os.path.join(PROBE_WEIGHTS_DIR, run_name)
os.makedirs(output_dir, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
parameters = {'condEntropyWeight': 1.0}

tl_probe = ambiProbe(cat_embs_info, "TinyLlama/TinyLlama-1.1B-Chat-v1.0", parameters)
from safetensors.torch import load_file
tl_probe.load_state_dict(load_file('./model/probes/TinyLlama_probe_3epoch_standard/model.safetensors'))
tl_probe.to(device)
tl_probe.eval()

condition_df = odf.copy()
# condition_df = odf[1:40]
eval_dataset = MTLDataset(
    condition_df, #onn full dataset
    cat_embs_info,
    tokenizer,
    max_length=128,
    zero_cats=False,
)
out_cat_names = eval_dataset.out_cat_names

result = evaluate_raw(
    tl_probe,
    eval_dataset,
    run_name, 
    batch_size=64,
    device=device,
)

row = one_row_summary(
    result,
    model_name=model_name,
    condition=condition,
)

try:
    giant_res = pd.read_csv(GIANT_RESULTS_DICT)
except:
    giant_res = pd.DataFrame()

giant_res = pd.concat(
    [giant_res, pd.DataFrame([row])],
    ignore_index=True,
)

giant_res.to_csv(GIANT_RESULTS_DICT, index=False)

write_eval_log(
    result,
    path=log_path,
    model_name=model_name,
    condition=condition,
    extra={
        "batch_size": 64,
        "seed": RANDOM_SEED,
        "n_eval_rows": len(condition_df),
    },
)

print_accuracy_section(
    by_head=result["by_head"],
    global_metrics=result["global"],
)

high_entropy = result["high_entropy"]
highest_percent = 100 * (1 - high_entropy["percentile"])

print_accuracy_section(
    by_head=high_entropy["by_head"],
    global_metrics=high_entropy["global"],
    title=(
        f"HIGHEST {highest_percent:.0f}% ENTROPY "
        f"(n={high_entropy['n_rows']}, "
        f"cutoff={high_entropy['threshold']:.4f})"
    ),
)

Evaluating probe: 100%|██████████| 778/778 [13:25<00:00,  1.04s/it]


Variable                     Probe    Marginal      Lift
───────────────────────────────────────────────────────
num_news_check_frequency     0.451       0.451    +0.000
num_foll_news_outlet         0.310       0.308    +0.001
num_poli_idealogy            0.385       0.357    +0.028
num_age                      0.753       0.753    +0.000
num_gender                   0.564       0.564    +0.000
num_education                0.819       0.819    +0.000
───────────────────────────────────────────────────────
GLOBAL                       0.547       0.542    +0.005

HIGHEST 25% ENTROPY (n=13608, cutoff=0.9544)
Variable                     Probe    Marginal      Lift
───────────────────────────────────────────────────────
num_news_check_frequency     0.442       0.442    +0.000
num_foll_news_outlet         0.308       0.302    +0.006
num_poli_idealogy            0.392       0.359    +0.032
num_age                      0.739       0.739    +0.000
num_gender                   0.562       0.56

In [9]:
#Test 2 - standard tinyllama 3 epoch (conditional entropy, shuffled outcats, no other changes), evaluated on f1/acc and lift
run_name = "TinyLlama_probe_3epoch_randCats"
model_name = "TinyLlama_probe_3epoch_randCats"
condition  = "none"
log_path = f"tests/{run_name}/full_log.json"

output_dir = os.path.join(PROBE_WEIGHTS_DIR, run_name)
os.makedirs(output_dir, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
parameters = {'condEntropyWeight': 1.0}

tl_probe = ambiProbe(cat_embs_info, "TinyLlama/TinyLlama-1.1B-Chat-v1.0", parameters)
from safetensors.torch import load_file
tl_probe.load_state_dict(load_file('./model/probes/TinyLlama_probe_3epoch_randCats/model.safetensors'))
tl_probe.to(device)
tl_probe.eval()

condition_df = odf.copy()
# condition_df = odf[1:40]
eval_dataset = MTLDataset(
    condition_df, #onn full dataset
    cat_embs_info,
    tokenizer,
    max_length=128,
    zero_cats=False,
)
out_cat_names = eval_dataset.out_cat_names

result = evaluate_raw(
    tl_probe,
    eval_dataset,
    run_name, 
    batch_size=64,
    device=device,
)

row = one_row_summary(
    result,
    model_name=model_name,
    condition=condition,
)

try:
    giant_res = pd.read_csv(GIANT_RESULTS_DICT)
except:
    giant_res = pd.DataFrame()

giant_res = pd.concat(
    [giant_res, pd.DataFrame([row])],
    ignore_index=True,
)

giant_res.to_csv(GIANT_RESULTS_DICT, index=False)

write_eval_log(
    result,
    path=log_path,
    model_name=model_name,
    condition=condition,
    extra={
        "batch_size": 64,
        "seed": RANDOM_SEED,
        "n_eval_rows": len(condition_df),
    },
)

print_accuracy_section(
    by_head=result["by_head"],
    global_metrics=result["global"],
)

high_entropy = result["high_entropy"]
highest_percent = 100 * (1 - high_entropy["percentile"])

print_accuracy_section(
    by_head=high_entropy["by_head"],
    global_metrics=high_entropy["global"],
    title=(
        f"HIGHEST {highest_percent:.0f}% ENTROPY "
        f"(n={high_entropy['n_rows']}, "
        f"cutoff={high_entropy['threshold']:.4f})"
    ),
)

Evaluating probe: 100%|██████████| 778/778 [13:27<00:00,  1.04s/it]


Variable                     Probe    Marginal      Lift
───────────────────────────────────────────────────────
num_news_check_frequency     0.451       0.451    -0.000
num_foll_news_outlet         0.301       0.308    -0.007
num_poli_idealogy            0.357       0.357    -0.000
num_age                      0.753       0.753    +0.000
num_gender                   0.561       0.564    -0.003
num_education                0.819       0.819    +0.000
───────────────────────────────────────────────────────
GLOBAL                       0.540       0.542    -0.002

HIGHEST 25% ENTROPY (n=13608, cutoff=0.9544)
Variable                     Probe    Marginal      Lift
───────────────────────────────────────────────────────
num_news_check_frequency     0.442       0.442    +0.000
num_foll_news_outlet         0.297       0.302    -0.005
num_poli_idealogy            0.358       0.359    -0.001
num_age                      0.739       0.739    +0.000
num_gender                   0.557       0.56

In [10]:
#Test 2 - standard tinyllama 3 epoch (conditional entropy, shuffled outcats, no other changes), evaluated on f1/acc and lift
run_name = "TinyLlama_probe_7epoch_standard"
model_name = "TinyLlama_probe_7epoch_standard"
condition  = "none"
log_path = f"tests/{run_name}/full_log.json"

output_dir = os.path.join(PROBE_WEIGHTS_DIR, run_name)
os.makedirs(output_dir, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
parameters = {'condEntropyWeight': 1.0}

tl_probe = ambiProbe(cat_embs_info, "TinyLlama/TinyLlama-1.1B-Chat-v1.0", parameters)
from safetensors.torch import load_file
tl_probe.load_state_dict(load_file('./model/probes/TinyLlama_probe_7epoch_standard/model.safetensors'))
tl_probe.to(device)
tl_probe.eval()

condition_df = odf.copy()
# condition_df = odf[1:40]
eval_dataset = MTLDataset(
    condition_df, #onn full dataset
    cat_embs_info,
    tokenizer,
    max_length=128,
    zero_cats=False,
)
out_cat_names = eval_dataset.out_cat_names

result = evaluate_raw(
    tl_probe,
    eval_dataset,
    run_name, 
    batch_size=64,
    device=device,
)

row = one_row_summary(
    result,
    model_name=model_name,
    condition=condition,
)

try:
    giant_res = pd.read_csv(GIANT_RESULTS_DICT)
except:
    giant_res = pd.DataFrame()

giant_res = pd.concat(
    [giant_res, pd.DataFrame([row])],
    ignore_index=True,
)

giant_res.to_csv(GIANT_RESULTS_DICT, index=False)

write_eval_log(
    result,
    path=log_path,
    model_name=model_name,
    condition=condition,
    extra={
        "batch_size": 64,
        "seed": RANDOM_SEED,
        "n_eval_rows": len(condition_df),
    },
)

print_accuracy_section(
    by_head=result["by_head"],
    global_metrics=result["global"],
)

high_entropy = result["high_entropy"]
highest_percent = 100 * (1 - high_entropy["percentile"])

print_accuracy_section(
    by_head=high_entropy["by_head"],
    global_metrics=high_entropy["global"],
    title=(
        f"HIGHEST {highest_percent:.0f}% ENTROPY "
        f"(n={high_entropy['n_rows']}, "
        f"cutoff={high_entropy['threshold']:.4f})"
    ),
)

Evaluating probe: 100%|██████████| 778/778 [13:25<00:00,  1.04s/it]


Variable                     Probe    Marginal      Lift
───────────────────────────────────────────────────────
num_news_check_frequency     0.449       0.451    -0.002
num_foll_news_outlet         0.291       0.308    -0.017
num_poli_idealogy            0.348       0.357    -0.009
num_age                      0.753       0.753    +0.000
num_gender                   0.549       0.564    -0.015
num_education                0.819       0.819    +0.000
───────────────────────────────────────────────────────
GLOBAL                       0.535       0.542    -0.007

HIGHEST 25% ENTROPY (n=13608, cutoff=0.9544)
Variable                     Probe    Marginal      Lift
───────────────────────────────────────────────────────
num_news_check_frequency     0.440       0.442    -0.002
num_foll_news_outlet         0.289       0.302    -0.013
num_poli_idealogy            0.345       0.359    -0.014
num_age                      0.739       0.739    +0.000
num_gender                   0.544       0.56

In [50]:
print('hello')

hello


## Testing Schedule
1. Train the tinyllama probe for three epochs, no modifications. Evaluate on the hold out setExamine simple accuracy and f1 counts overall.
2. Evaluate this probe on shuffled labels and shuffled targets, ensure make sense intuitively. 
3. Make necessary changes until we're happy with this. 
4. Run the same tinyllama for 7,10 epohcs, no modifications. evaluate on hold out set only, determine optimal epochs. 
5. For the best tinyllama, run inference on the entire dataset - both train and test on shuffled labels and targets, also compare pieceswise performance not just overall.
6. Run inference on the 25% high quartile of entropy rows, and 50%. 
7. Train the qwen and deberta probes, ensure performance similar to tinyllama